# Day 8 — Clean Code & SOLID Principles Applied

## Objective
Demonstrate the Day 8 refactoring work based on `REFACTOR_NOTES.md`. Highlight Single Responsibility Principle (SRP), Open/Closed Principle (OCP), Dependency Inversion Principle (DIP), Composition over Inheritance, DRY helper extraction, and verify zero source code modification on `Pipeline` when adding new steps.

## 1. Refactoring Summary (SOLID Principles)

- **SRP**: Extracted `DataLoader` from `Pipeline` to separate data retrieval/retries from step execution.
- **OCP & DIP**: Created `RemoveDuplicatesStep(Step)`. Added it to `Pipeline` without modifying `Pipeline` source code.
- **DRY**: Extracted `Pipeline._count_records` static helper method.

In [1]:
import sys
import inspect
from pathlib import Path
repo_root = Path.cwd().resolve()
sys.path.insert(0, str(repo_root / 'src'))

from task_analytics import (
    Pipeline, DataLoader, CleanDataStep, NormalizeDataStep, RemoveDuplicatesStep, Step
)
print('Refactored Day 8 components loaded successfully.')

Refactored Day 8 components loaded successfully.


## 2. SRP Demonstration: `DataLoader` vs `Pipeline`

`DataLoader` handles remote data retries independently from `Pipeline` execution.

In [2]:
attempts = 0
def mock_fetcher():
    global attempts
    attempts += 1
    if attempts < 2:
        print('  DataLoader: Simulated fetch retry...')
        raise ConnectionError('Glitch')
    return [{'id': 1, 'title': 'Task A'}, {'id': 2, 'title': 'Task B'}]

loader = DataLoader(max_retries=3)
fetched = loader.load(mock_fetcher)
print('DataLoader retrieved:', len(fetched), 'items')

  DataLoader: Simulated fetch retry...
DataLoader retrieved: 2 items


## 3. OCP & DIP Demonstration: New `RemoveDuplicatesStep`

Add `RemoveDuplicatesStep` to `Pipeline` and verify `Pipeline` class source code was **NOT modified**.

In [3]:
raw_data = [
    {'id': 1, 'title': ' Fix Login Bug ', 'status': ' IN_PROGRESS '},
    {'id': 2, 'title': 'Write Tests', 'status': 'PENDING'},
    {'id': 1, 'title': ' Fix Login Bug ', 'status': ' IN_PROGRESS '},  # duplicate id 1
]

pipeline_src_before = inspect.getsource(Pipeline)

pipeline = Pipeline([
    CleanDataStep(required_keys=['id', 'title']),
    NormalizeDataStep(target_fields=['status']),
])

# Add new step dynamically (Composition)
dedup_step = RemoveDuplicatesStep(key='id')
pipeline.add_step(dedup_step)

output = pipeline.run(raw_data)
pipeline_src_after = inspect.getsource(Pipeline)

assert pipeline_src_before == pipeline_src_after, 'Pipeline class source was modified!'
print(f'Pipeline output count after deduplication: {len(output)}')
for item in output:
    print('  ->', item)
print('\n[VERIFICATION SUCCESS]: Pipeline source code was NOT modified when adding RemoveDuplicatesStep!')

Pipeline output count after deduplication: 2
  -> {'id': 1, 'title': ' Fix Login Bug ', 'status': 'in_progress'}
  -> {'id': 2, 'title': 'Write Tests', 'status': 'pending'}

[VERIFICATION SUCCESS]: Pipeline source code was NOT modified when adding RemoveDuplicatesStep!


## Conclusion & Key Takeaways

- **SRP**: Dedicated classes (`DataLoader`, `Pipeline`, concrete `Step`s) reduce coupling.
- **OCP/DIP**: High-level modules depending on abstract interfaces (`Step`) enable endless extension without source code modification.